<a href="https://colab.research.google.com/github/yenlikgaisina-ux/construction-sustainability-agent/blob/main/notebook/construction_sustainability_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Construction Sustainability Research Agent

**Author:** Yenlik Gaisina | Data & Analytics Consultant | Cambridge Data Science with ML & AI
**Framework:** CrewAI | LangChain | Google Gemini
**Methods:** Multi-Agent Systems | Agentic AI | Prompt Engineering | Web Search Automation
**Objective:** Automate Scope 3 and embodied carbon research for UK construction sustainability managers using a three-agent CrewAI pipeline

---

## Table of Contents

1. [Business Context & Problem Statement](#1-business-context)
2. [Architecture Overview](#2-architecture)
3. [Installation & Setup](#3-setup)
4. [Tool Definitions](#4-tools)
5. [Agent Definitions](#5-agents)
6. [Task Definitions](#6-tasks)
7. [Crew Assembly & Execution](#7-crew)
8. [Final Briefing Output](#8-output)
9. [Execution Performance](#9-performance)
10. [Reflection & Extension](#10-reflection)

## 1. Business Context

### Problem Statement

UK construction firms face a growing compliance burden: mandatory whole-life carbon reporting under the UK Net Zero Carbon Buildings Standard, Scope 3 disclosure requirements in CDP and TCFD frameworks, and BREEAM certification pressures from clients. A sustainability manager at a mid-size contractor currently spends **2-3 days** per month manually researching regulatory updates, benchmarking embodied carbon figures, and synthesising guidance from UKGBC, RICS, GHG Protocol, and BREEAM documentation.

> **Business Question:** Can a multi-agent AI system autonomously research, analyse, and synthesise current Scope 3 and embodied carbon intelligence -- reducing research time from days to under 10 minutes, while maintaining professional accuracy?

### Why Multi-Agent?

A single LLM prompt cannot reliably research, validate, and write simultaneously -- it either hallucinates sources or produces generic output. The **CrewAI multi-agent pattern** solves this by:

- **Separation of concerns**: each agent has one specialised role and cannot stray outside it
- **Sequential validation**: the Analyst receives the Researcher's raw findings and must verify them before passing to the Writer
- **Tooling isolation**: the Researcher uses search tools; the Analyst processes data without external access; the Writer has no external tools, forcing synthesis from validated inputs only
- **Auditability**: the full execution trace is logged, showing exactly which sources were consulted and which figures were retained vs. discarded

### Demonstration Topic

This notebook runs the agent crew on a specific, realistic brief:
**"Scope 3 Category 1 (Purchased Goods & Services) reporting requirements and benchmarks for UK construction subcontractors, 2024 reporting cycle"**

## 2. Architecture Overview

```
USER INPUT: Research Topic
      |
      v
+--------------------+    SerperDev Search
| RESEARCHER AGENT   |--> web_search()      --> source URLs + excerpts
| Senior Sustainabi- |    scrape_url()      --> raw page content
| lity Researcher    |    site filter:      --> UKGBC, RICS, GHG
+--------------------+       Protocol, BREEAM, gov.uk
      |
      | raw findings package
      v
+--------------------+
| ANALYST AGENT      |--> cross-reference   --> metrics table
| Carbon Data        |    validate figures  --> source confidence
| Analyst            |    flag gaps         --> data quality notes
+--------------------+
      |
      | validated analysis + structured metrics
      v
+--------------------+
| WRITER AGENT       |    (no external tools - synthesis only)
| Technical Report   |--> Executive Summary
| Writer             |    Regulatory Context
|                    |    Key Findings + metrics table
+--------------------+    Recommended Actions (3-5 items)
      |
      v
OUTPUT: Professional Markdown Briefing (800-1200 words)
```

**Process:** Sequential (Researcher -> Analyst -> Writer). Each agent receives the full context of all previous agents' outputs.

## 3. Installation & Setup

In [ ]:
# Install required packages
!pip install -q --upgrade crewai crewai-tools langchain-google-genai langchain-core

In [ ]:
import os
import json
import time
import warnings
import textwrap
from datetime import datetime

warnings.filterwarnings('ignore')

from crewai import Agent, Task, Crew, Process
from crewai_tools import SerperDevTool, ScrapeWebsiteTool
from langchain_google_genai import ChatGoogleGenerativeAI

print('CrewAI import successful')
print(f'Notebook run: {datetime.now().strftime("%Y-%m-%d %H:%M")}')

In [ ]:
# ── API Key Configuration ─────────────────────────────────────────────
# Required: Google API key (Gemini) + SerperDev API key (web search)
#
# Option 1: Colab Secrets (recommended) -- click the key icon in sidebar
# Option 2: Environment variables: GOOGLE_API_KEY, SERPER_API_KEY
# ──────────────────────────────────────────────────────────────────────

try:
    from google.colab import userdata
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
    SERPER_API_KEY = userdata.get('SERPER_API_KEY')
    os.environ['GOOGLE_API_KEY'] = GOOGLE_API_KEY
    os.environ['SERPER_API_KEY'] = SERPER_API_KEY
    print('API keys loaded from Colab Secrets')
except Exception:
    GOOGLE_API_KEY = os.environ.get('GOOGLE_API_KEY', '')
    SERPER_API_KEY = os.environ.get('SERPER_API_KEY', '')
    if GOOGLE_API_KEY and SERPER_API_KEY:
        print('API keys loaded from environment variables')
    else:
        raise ValueError(
            'API keys not found. Set GOOGLE_API_KEY and SERPER_API_KEY '
            'in Colab Secrets or as environment variables.'
        )

# ── LLM Configuration ────────────────────────────────────────────────
# Gemini 2.0 Flash: fast, cost-effective, strong instruction-following
llm = ChatGoogleGenerativeAI(
    model='gemini-2.0-flash',
    temperature=0.2,
    max_output_tokens=4096,
    google_api_key=GOOGLE_API_KEY,
)
print(f'LLM configured: {llm.model}')

## 4. Tool Definitions

In [ ]:
# ── Search and Scraping Tools ─────────────────────────────────────────
# SerperDev: Google Search API for live web queries
# ScrapeWebsiteTool: extracts clean text from target URLs
# ──────────────────────────────────────────────────────────────────────

search_tool = SerperDevTool(n_results=10)
scrape_tool = ScrapeWebsiteTool()

# Authoritative source domains the Researcher is instructed to prioritise
PRIORITY_DOMAINS = [
    'ukgbc.org',
    'rics.org',
    'breeam.com',
    'leti.uk',
    'ghgprotocol.org',
    'gov.uk',
    'ice.org.uk',
    'constructionleadershipcouncil.co.uk',
    'istructe.org',
]

print(f'Search tool : SerperDev (n_results=10)')
print(f'Scrape tool : ScrapeWebsiteTool')
print(f'Priority domains: {len(PRIORITY_DOMAINS)}')

## 5. Agent Definitions

In [ ]:
# ── AGENT 1: RESEARCHER ───────────────────────────────────────────────
# Role: Surface current, authoritative sustainability data from the web
# Tools: SerperDev search + web scraper
# ──────────────────────────────────────────────────────────────────────

researcher = Agent(
    role='Senior Sustainability Researcher',
    goal=(
        'Search the web to find current, authoritative data on the given '
        'sustainability topic. Focus exclusively on UK construction context. '
        'Prioritise sources from UKGBC, RICS, BREEAM, GHG Protocol, LETI, '
        'and UK Government. Return a structured package of source URLs with '
        'key verbatim excerpts and numeric data.'
    ),
    backstory=(
        'You are a senior sustainability researcher with 12 years of experience '
        'advising Tier 1 UK contractors on carbon reporting, BREEAM certification, '
        'and supply chain emissions. You are rigorous about source quality -- you '
        'never cite secondary sources when primary regulatory guidance is available. '
        'You have deep knowledge of the GHG Protocol Corporate Value Chain '
        '(Scope 3) Standard and its application to UK construction procurement.'
    ),
    tools=[search_tool, scrape_tool],
    llm=llm,
    verbose=True,
    allow_delegation=False,
    max_iter=5,
)
print('Agent 1 configured: Senior Sustainability Researcher')

In [ ]:
# ── AGENT 2: ANALYST ──────────────────────────────────────────────────
# Role: Validate findings, extract metrics, identify data gaps
# Tools: None (works from Researcher output only)
# ──────────────────────────────────────────────────────────────────────

analyst = Agent(
    role='Carbon Data Analyst',
    goal=(
        'Receive the Researcher findings package and perform critical analysis. '
        'Cross-reference figures across multiple sources to validate accuracy. '
        'Build a structured metrics table with: metric name, value, unit, source, '
        'and confidence level (High/Medium/Low). '
        'Identify any data gaps or contradictions and flag them explicitly.'
    ),
    backstory=(
        'You are a quantitative carbon analyst who previously worked at the Carbon '
        'Disclosure Project (CDP) and now consults for major UK infrastructure clients. '
        'You are trained to spot hallucinated or outdated data immediately. '
        'You always produce structured, table-formatted output with explicit '
        'source attribution and confidence ratings. You flag when figures '
        'come from a single source as lower confidence.'
    ),
    tools=[],
    llm=llm,
    verbose=True,
    allow_delegation=False,
    max_iter=4,
)
print('Agent 2 configured: Carbon Data Analyst')

In [ ]:
# ── AGENT 3: WRITER ───────────────────────────────────────────────────
# Role: Synthesise validated analysis into a professional briefing
# Tools: None (synthesis from validated inputs only)
# ──────────────────────────────────────────────────────────────────────

writer = Agent(
    role='Technical Report Writer',
    goal=(
        'Write a professional sustainability briefing (800-1200 words) using only '
        'the validated analysis provided by the Analyst. '
        'Structure: Executive Summary, Regulatory Context, Key Findings (with '
        'metrics table), and Recommended Actions (3-5 concrete steps). '
        'Tone: authoritative, plain English, suitable for a project director '
        'with no carbon accounting background. No jargon without definition.'
    ),
    backstory=(
        'You are a technical writer with 10 years of experience producing '
        'sustainability reports, bid documents, and policy briefings for major UK '
        'construction firms. You have written submissions for BREEAM assessments, '
        'CDP climate questionnaires, and ESG investor briefings. Your writing is '
        'precise, structured, and always leads with the business implication rather '
        'than the methodology.'
    ),
    tools=[],
    llm=llm,
    verbose=True,
    allow_delegation=False,
    max_iter=3,
)
print('Agent 3 configured: Technical Report Writer')
print(f'\nAll {len([researcher, analyst, writer])} agents initialised.')

## 6. Task Definitions

In [ ]:
# ── Research topic (change this to run on different topics) ───────────
RESEARCH_TOPIC = (
    "Scope 3 Category 1 (Purchased Goods and Services) reporting requirements "
    "and embodied carbon benchmarks for UK construction subcontractors, "
    "2024 reporting cycle. Include GHG Protocol guidance, RICS PS3 requirements, "
    "and UKGBC Net Zero Carbon Buildings Standard targets."
)

print(f'Research topic:\n{RESEARCH_TOPIC}')

In [ ]:
# ── TASK 1: Research ──────────────────────────────────────────────────
task_research = Task(
    description=(
        f'Research the following topic comprehensively:\n\n{RESEARCH_TOPIC}\n\n'
        'Requirements:\n'
        '1. Use SerperDev to search for at least 5 distinct queries covering '
        '   regulatory, benchmark, and practical guidance angles\n'
        '2. Scrape the top 3-5 most authoritative URLs to extract verbatim data\n'
        '3. Return a structured findings package with: source_url, excerpt, '
        '   key_metric (if numeric), publication_date, confidence\n'
        '4. Minimum 8 distinct sources; at least 3 must be primary regulatory guidance'
    ),
    expected_output=(
        'A structured findings package containing 8-12 source entries, each with: '
        'URL, key excerpt (verbatim), numeric metrics where available, '
        'publication year, and confidence assessment (High/Medium/Low).'
    ),
    agent=researcher,
)

# ── TASK 2: Analysis ─────────────────────────────────────────────────
task_analysis = Task(
    description=(
        'You will receive the Researcher findings package. Your job is to:\n'
        '1. Cross-reference all numeric figures -- flag if the same metric varies '
        '   significantly across sources (>20% variance = flag as disputed)\n'
        '2. Build a structured metrics table: '
        '   Metric | Value | Unit | Source | Year | Confidence\n'
        '3. Identify the 3 most important compliance deadlines\n'
        '4. Calculate the estimated Scope 3 Category 1 share for a typical '
        '   GBP 50M UK construction project using average industry ratios\n'
        '5. Flag any data gaps where the Researcher found no authoritative source'
    ),
    expected_output=(
        'A validated metrics table (Markdown format), list of 3 key compliance '
        'deadlines, estimated Scope 3 Category 1 figure for a GBP 50M project, '
        'and a data gaps section.'
    ),
    agent=analyst,
    context=[task_research],
)

# ── TASK 3: Write Briefing ───────────────────────────────────────────
task_write = Task(
    description=(
        'Using the Analyst validated metrics and analysis, write a professional '
        'sustainability briefing for a UK construction project director. Structure:\n'
        '1. EXECUTIVE SUMMARY (150 words max): What does this mean for our business TODAY?\n'
        '2. REGULATORY CONTEXT (200 words): What must we comply with and by when?\n'
        '3. KEY FINDINGS (use the metrics table from the Analyst): What do the numbers show?\n'
        '4. RECOMMENDED ACTIONS (3-5 concrete actions with owners and timelines)\n'
        'Style: authoritative, no passive voice, lead with impact. '
        'Define any technical terms on first use.'
    ),
    expected_output=(
        'A complete professional sustainability briefing in Markdown format, '
        '800-1200 words, with all four sections, metrics table, and action items.'
    ),
    agent=writer,
    context=[task_research, task_analysis],
)

print('Tasks defined:')
for i, t in enumerate([task_research, task_analysis, task_write], 1):
    print(f'  Task {i}: {t.agent.role}')

## 7. Crew Assembly & Execution

In [ ]:
# ── Crew Assembly ─────────────────────────────────────────────────────
# Process.sequential: agents run in order (Researcher -> Analyst -> Writer)
# Each agent receives the full context of previous agents' outputs
# ──────────────────────────────────────────────────────────────────────

crew = Crew(
    agents=[researcher, analyst, writer],
    tasks=[task_research, task_analysis, task_write],
    process=Process.sequential,
    verbose=True,
    memory=False,
)

print(f'Crew assembled: {len(crew.agents)} agents, {len(crew.tasks)} tasks')
print(f'Process: {crew.process}')
print('=' * 60)
print('Starting crew execution...\n')

t_start = time.time()
result = crew.kickoff()
t_end = time.time()

elapsed = t_end - t_start
print('\n' + '=' * 60)
print(f'Crew execution complete in {elapsed:.0f} seconds ({elapsed/60:.1f} min)')

## 8. Final Briefing Output

The Writer agent produced the following professional sustainability briefing based on the Researcher and Analyst outputs.

In [ ]:
# ── Display the final briefing ────────────────────────────────────────
from IPython.display import Markdown, display

display(Markdown(str(result)))

## 9. Execution Performance

Timing and output metrics captured from the live run above.

In [ ]:
# ── Performance metrics from the live run ─────────────────────────────
briefing_text = str(result)
word_count = len(briefing_text.split())

print('EXECUTION PERFORMANCE')
print('=' * 40)
print(f'Total run time     : {elapsed:.0f} sec ({elapsed/60:.1f} min)')
print(f'Briefing length    : {word_count} words')
print(f'LLM                : {llm.model}')
print(f'Agents             : {len(crew.agents)}')
print(f'Tasks              : {len(crew.tasks)}')
print(f'Process            : {crew.process}')
print(f'Manual equivalent  : ~2-3 days researcher time')

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

plt.style.use('dark_background')

NAVY  = '#0a2342'
BLUE  = '#1a5276'
GREEN = '#2D6A4F'
LIGHT = '#52b788'
WARM  = '#e9c46a'
RED   = '#e76f51'
WHITE = '#f8f9fa'
MUTED = '#adb5bd'

fig, ax = plt.subplots(figsize=(10, 7))
fig.patch.set_facecolor('#0d1117')
ax.set_facecolor('#0d1117')
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.axis('off')

# Title
ax.text(5, 9.5, 'CrewAI Agent Pipeline', fontsize=16, fontweight='bold',
        color=WHITE, ha='center', va='center')

# Input box
input_box = mpatches.FancyBboxPatch((3.5, 8.3), 3, 0.7, boxstyle='round,pad=0.1',
                                     facecolor=MUTED, edgecolor=WHITE, linewidth=1.5)
ax.add_patch(input_box)
ax.text(5, 8.65, 'Research Topic', fontsize=10, fontweight='bold',
        color='#0d1117', ha='center', va='center')

# Arrow 1
ax.annotate('', xy=(5, 7.8), xytext=(5, 8.3),
            arrowprops=dict(arrowstyle='->', color=MUTED, lw=2))

# Researcher box
r_box = mpatches.FancyBboxPatch((2.5, 6.5), 5, 1.2, boxstyle='round,pad=0.15',
                                 facecolor=BLUE, edgecolor=WHITE, linewidth=1.5)
ax.add_patch(r_box)
ax.text(5, 7.3, 'RESEARCHER AGENT', fontsize=11, fontweight='bold',
        color=WHITE, ha='center', va='center')
ax.text(5, 6.9, 'SerperDev Search + Web Scraping', fontsize=8,
        color=MUTED, ha='center', va='center')

# Arrow 2
ax.annotate('', xy=(5, 5.2), xytext=(5, 6.5),
            arrowprops=dict(arrowstyle='->', color=MUTED, lw=2))
ax.text(7.8, 5.85, 'raw findings\n(URLs, excerpts, data)', fontsize=7,
        color=MUTED, ha='center', va='center', style='italic')

# Analyst box
a_box = mpatches.FancyBboxPatch((2.5, 4.0), 5, 1.2, boxstyle='round,pad=0.15',
                                 facecolor=GREEN, edgecolor=WHITE, linewidth=1.5)
ax.add_patch(a_box)
ax.text(5, 4.8, 'ANALYST AGENT', fontsize=11, fontweight='bold',
        color=WHITE, ha='center', va='center')
ax.text(5, 4.4, 'Cross-reference + Validate + Flag Gaps', fontsize=8,
        color=MUTED, ha='center', va='center')

# Arrow 3
ax.annotate('', xy=(5, 2.7), xytext=(5, 4.0),
            arrowprops=dict(arrowstyle='->', color=MUTED, lw=2))
ax.text(7.8, 3.35, 'validated metrics\n+ confidence ratings', fontsize=7,
        color=MUTED, ha='center', va='center', style='italic')

# Writer box
w_box = mpatches.FancyBboxPatch((2.5, 1.5), 5, 1.2, boxstyle='round,pad=0.15',
                                 facecolor=WARM, edgecolor=WHITE, linewidth=1.5)
ax.add_patch(w_box)
ax.text(5, 2.3, 'WRITER AGENT', fontsize=11, fontweight='bold',
        color='#0d1117', ha='center', va='center')
ax.text(5, 1.9, 'Synthesis Only (no external tools)', fontsize=8,
        color='#0d1117', ha='center', va='center')

# Arrow 4
ax.annotate('', xy=(5, 0.7), xytext=(5, 1.5),
            arrowprops=dict(arrowstyle='->', color=MUTED, lw=2))

# Output box
out_box = mpatches.FancyBboxPatch((2.8, 0.1), 4.4, 0.6, boxstyle='round,pad=0.1',
                                   facecolor=LIGHT, edgecolor=WHITE, linewidth=1.5)
ax.add_patch(out_box)
ax.text(5, 0.4, 'Professional Briefing (800-1200 words)', fontsize=9,
        fontweight='bold', color='#0d1117', ha='center', va='center')

plt.tight_layout()
plt.show()
print('Pipeline architecture diagram rendered')

## 10. Reflection & Extension

### What Worked Well

The sequential CrewAI pipeline performed reliably on UK construction sustainability topics. Key strengths:

- **Source validation**: The Analyst's cross-referencing step catches inconsistencies between sources before the Writer synthesises them. On well-regulated topics (Scope 3, BREEAM), the high alignment between UKGBC, RICS, and CDP data confirms source quality. On less-regulated topics, the Analyst flags disputes in numeric claims -- exactly the behaviour it was designed for.
- **Structured output**: All four briefing sections (Executive Summary, Regulatory Context, Key Findings, Recommended Actions) are produced to specification, including metrics tables with source attribution and confidence levels.
- **Tooling isolation**: The Writer has no external tools, which forces it to synthesise only from the Analyst's validated output rather than introducing unverified claims.

### Limitations

- **Snippet vs. full document**: SerperDev returns search snippets, not full documents. The Researcher can scrape URLs for full content, but occasionally cites snippet text without full verification. Mitigation: enforce mandatory scrape for all regulatory sources.
- **UK-specific data gaps**: Supplier-specific Scope 3 Category 1 data for UK Tier 2/3 subcontractors is genuinely scarce. The spend-based EEIO methodology is the practical ceiling for most firms.
- **Temporal drift**: Carbon market prices and EEIO factors change annually. The system should be scheduled to re-run periodically rather than treated as a one-time output.

### Potential Extensions

1. **PDF Ingestion Agent**: A fourth agent with a PDF reader tool to process uploaded project specifications and extract project-specific parameters (floor area, material schedules) before the Researcher runs.
2. **Supplier Benchmarking**: Query supplier sustainability disclosures from CDP's open API and compare them against industry benchmarks.
3. **Scheduled Monitoring**: Deploy as a CrewAI flow triggered weekly to monitor for new RICS, UKGBC, or GHG Protocol guidance updates.
4. **BREEAM Mat01 Calculator**: Integrate the BREEAM Materials calculator logic as a custom tool so the Analyst can estimate Mat01 credit scores directly from material specifications.

### Sample Research Topics Tested

1. Scope 3 Category 1: Purchased Goods & Services for UK general contractors (2024 reporting cycle)
2. Embodied carbon benchmarks for concrete-frame commercial buildings (RICS/LETI targets)
3. Whole-life carbon assessment requirements under UK Net Zero Carbon Buildings Standard
4. BREEAM UK New Construction 2018: Materials and Waste credits pathway
5. Circular economy procurement strategies for demolition and fit-out waste